# GMM-NHMM — sub-modes inside each regime

Some regimes aren't unimodal. A bear market has both *slow grinding declines* and *panic crashes*; a bull market has *smooth uptrend* and *FOMO spike*. A single Gaussian per regime smooths the two together and loses information.

**GMM-NHMM** keeps the NHMM machinery (covariate-dependent transitions between regimes) and replaces each regime's single Gaussian with a `n_mix`-component Gaussian mixture — letting each regime express internal sub-modes.

**This notebook** : simulate 2 regimes × 2 sub-modes, fit `fit_gmm_nhmm`, inspect the per-state mixtures and how they're modulated by the covariate.

## 1. Simulate 2 regimes × 2 sub-modes each

Regime 0 (calm) mixes a low-vol sub-mode and a mean-reverting sub-mode. Regime 1 (volatile) mixes a panic-down sub-mode and a FOMO-up sub-mode. A scalar covariate (think: realized vol or funding rate) drives transitions between the two regimes.

In [ ]:
import numpy as np

rng = np.random.default_rng(123)
T = 1500
Z = rng.normal(0, 1, (T, 1))   # one scalar covariate

means = np.array([
    [[-1.0, -1.0], [ 1.0,  1.0]],   # regime 0 sub-modes
    [[ 4.0,  4.0], [-4.0, -4.0]],   # regime 1 sub-modes
])
cov = 0.3 * np.eye(2)

state = 0
X = np.zeros((T, 2))
for t in range(T):
    sub = rng.integers(0, 2)
    X[t] = rng.multivariate_normal(means[state, sub], cov)
    p_stay = 0.92 if Z[t, 0] < 0 else 0.55   # high Z destabilizes the regime
    state = state if rng.random() < p_stay else 1 - state

X.shape, Z.shape

## 2. Declare a GMM-NHMM topology and fit

`EmissionSpec(type='gmm', n_mix=2)` tells the engine each state hosts a 2-component Gaussian mixture. `fit_gmm_nhmm` runs the 2-stage decomposition : GMM-HMM on `X`, then per-state logistic regression of the next state on `Z`.

In [ ]:
from hmm_core.topology import Topology, EmissionSpec, FitSpec, InitSpec
from hmm_core.gmm_nhmm import fit_gmm_nhmm

topo = Topology(
    name="gmm_nhmm_2regime_2submode",
    n_states=2,
    state_names=["calm", "volatile"],
    emission=EmissionSpec(
        type="gmm", covariance_type="diag", n_features=2, n_mix=2,
    ),
    allowed_transitions=None,
    startprob="uniform",
    init=InitSpec(strategy="kmeans", seed=42),
    fit=FitSpec(algorithm="baum_welch", n_iter=100, tol=1e-4),
)

result = fit_gmm_nhmm(topo, X, Z, covariate_names=["vol_proxy"], seed=42)
result

## 3. Compare against a single-Gaussian NHMM

BIC penalises model complexity. If the data really has sub-modes, GMM-NHMM (more parameters) should still win on BIC.

In [ ]:
from hmm_core.nhmm import fit_nhmm

topo_gaussian = Topology(
    name="gaussian_nhmm_2regime",
    n_states=2,
    state_names=["calm", "volatile"],
    emission=EmissionSpec(type="gaussian", covariance_type="diag", n_features=2),
    allowed_transitions=None,
    startprob="uniform",
    init=InitSpec(strategy="kmeans", seed=42),
    fit=FitSpec(algorithm="baum_welch", n_iter=100, tol=1e-4),
)
result_gauss = fit_nhmm(topo_gaussian, X, Z, covariate_names=["vol_proxy"], seed=42)

print(f"GMM-NHMM (n_mix=2) : BIC = {result.base.bic:.1f}, AIC = {result.base.aic:.1f}")
print(f"Gaussian NHMM      : BIC = {result_gauss.base.bic:.1f}, AIC = {result_gauss.base.aic:.1f}")
print()
winner = "GMM-NHMM" if result.base.bic < result_gauss.base.bic else "Gaussian NHMM"
print(f"Winner by BIC : {winner}")

## 4. Inspect the fitted sub-modes

Each state's mixture means should recover roughly the simulation values (up to label permutation). The HTML view above already shows this; here's the raw arrays for cross-checking.

In [ ]:
model = result.base.model
for state_idx in range(model.n_components):
    print(f"State {state_idx} ({topo.state_names[state_idx]}) sub-mode means :")
    for mix_idx in range(model.n_mix):
        print(f"  sub-mode {mix_idx} : {model.means_[state_idx, mix_idx]}")
    print()

## 5. Time-varying transitions still work

`A_at(t)` returns the 2×2 transition matrix at time `t`. With the covariate-driven regime switching above, low-Z timesteps should have higher diagonal (sticky), high-Z timesteps lower.

In [ ]:
low_t = int(np.argmin(Z[:, 0]))
high_t = int(np.argmax(Z[:, 0]))
print(f"At t={low_t} (low vol, z={Z[low_t, 0]:.2f}) :")
print(result.A_at(low_t).round(3))
print(f"\nAt t={high_t} (high vol, z={Z[high_t, 0]:.2f}) :")
print(result.A_at(high_t).round(3))

## Next

- **Factorial NHMM** : when you have multiple independent regime dimensions (trend × volatility × macro), see `06_factorial_nhmm_multifactor.ipynb`.
- **GMM-NHMM design** : full rationale in `docs/decisions/0007-gmm-nhmm-scope.md` (2-stage vs joint EM trade-off).